In [20]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pywt
from pylab import mpl
# 设置中文显示字体
mpl.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams['figure.constrained_layout.use'] = False # 禁用

In [21]:
#加载数据
def data_load(file_path):
    df=pd.read_csv(file_path,usecols=['CH17'])
    df=df.iloc[:,0].values
    return df

In [22]:
data_normal_20hz=data_load('../../数据集/BJTU/正常/data_leftaxlebox_M0_G0_LA0_RA0_20Hz_-10kN.csv')
data_inner_20hz=data_load('../../数据集/BJTU/内圈/data_leftaxlebox_M0_G0_LA1_RA0_20Hz_-10kN.csv')
data_outer_20hz=data_load('../../数据集/BJTU/外圈/data_leftaxlebox_M0_G0_LA2_RA0_20Hz_-10kN.csv')
data_rolling_20hz=data_load('../../数据集/BJTU/滚动体/data_leftaxlebox_M0_G0_LA3_RA0_20Hz_-10kN.csv')
data_inner_outer_20hz=data_load('../../数据集/BJTU/外圈加内圈/data_leftaxlebox_M0_G0_LA1+LA2_RA0_20Hz_-10kN.csv')
data_outer_rolling_20hz=data_load('../../数据集/BJTU/外圈加滚动体/data_leftaxlebox_M0_G0_LA2+LA3_RA0_20Hz_-10kN.csv')

In [23]:
def wav_trans(fs,wavelet_name,data):
    # 选择小波基函数：这里选用复 Morlet 小波 ('cmor')，它在故障诊断中非常经典
    
    # 定义我们要观察的尺度 (Scales)。尺度越小，对应频率越高；尺度越大，对应频率越低。
    scales = np.arange(1, 128)
    
    # 执行连续小波变换
    # cwtmatr 包含了变换后的复数矩阵（即时频特征），frequencies 是对应的实际频率
    coefficients, frequencies = pywt.cwt(data, scales, wavelet_name, sampling_period=1/fs)
    
    # 取绝对值（振幅），用于可视化
    amplitude_scalogram = np.abs(coefficients)
    return  frequencies,amplitude_scalogram

In [24]:
fs=64000 #fs 采样频率

In [25]:
import cv2  # 用于调整图像尺寸
import numpy as np

def create_samples(data, sample_length=8192, num_samples=100):
    """
    原论文数据集是10k的采样频率，采样时长为30s,我采用的BJTU数据集采样频率64k,采样时长10s,所以采用更大的采样频率8192，但是考虑到大窗口会导致样本数减少，因此使用重叠滑动窗口
    """
    samples = []
    total_length = len(data)
    
    # 如果数据总量连一个窗口都不够，直接报错
    if total_length < sample_length:
        raise ValueError(f"数据总长度 ({total_length}) 小于设定的窗口长度 ({sample_length})")
    # 计算每次滑动的步长
    # 如果要求的样本数为1，步长设为0即可；否则按公式计算步长
    if num_samples > 1:
        step_size = (total_length - sample_length) // (num_samples - 1)
    else:
        step_size = 0
    for i in range(num_samples):
        start_idx = i * step_size
        end_idx = start_idx + sample_length
        
        # 防止因取整带来的数组越界
        if end_idx > total_length:
            break
        samples.append(data[start_idx:end_idx])
    return np.array(samples)



def preprocess_pipeline(samples, fs, wavelet_name='morl'):
    processed_images = []
    for seq in samples:
        # 1. 连续小波变换获取时频矩阵
        _, amp = wav_trans(fs, wavelet_name, seq)

        # 2. Resize 尺寸调整
        # cv2.resize 参数格式为 (width, height)，对应论文中的 48 列 64 行
        img_resized = cv2.resize(amp, (48, 64), interpolation=cv2.INTER_LINEAR)

        # 3. Z-score 标准化
        img_normalized = (img_resized - np.mean(img_resized)) / np.std(img_resized)

        # 4. 扩展为 3 通道 (匹配论文中网络 Input size: [3, 64, 48])
        # 简单复制三个通道，满足 PyTorch Conv2d 的三通道输入要求
        img_3_channel = np.stack((img_normalized,) * 3, axis=0)

        processed_images.append(img_3_channel)

    return np.array(processed_images)  # 输出形状: (100, 3, 64, 48)


# 切分样本 (每类提取 100 个样本，每个样本 8192 个点)
samples_normal = create_samples(data_normal_20hz)
samples_inner = create_samples(data_inner_20hz)
samples_outer = create_samples(data_outer_20hz)
samples_rolling = create_samples(data_rolling_20hz)
samples_io = create_samples(data_inner_outer_20hz)
samples_or = create_samples(data_outer_rolling_20hz)

# 执行 CWT 及图像预处理（该过程可能需要一点计算时间）
print("正在生成时频特征图...")
X_normal = preprocess_pipeline(samples_normal, fs)
X_inner = preprocess_pipeline(samples_inner, fs)
X_outer = preprocess_pipeline(samples_outer, fs)
X_rolling = preprocess_pipeline(samples_rolling, fs)
X_io = preprocess_pipeline(samples_io, fs)
X_or = preprocess_pipeline(samples_or, fs)

# 标签向量定义：[内圈故障, 外圈故障, 滚动体故障]
y_normal = np.tile([0, 0, 0], (100, 1))  # 正常
y_inner = np.tile([1, 0, 0], (100, 1))  # 内圈单故障
y_outer = np.tile([0, 1, 0], (100, 1))  # 外圈单故障
y_rolling = np.tile([0, 0, 1], (100, 1))  # 滚动体单故障

# 复合故障标签（多个位置置 1）
y_io = np.tile([1, 1, 0], (100, 1))  # 内圈 + 外圈
y_or = np.tile([0, 1, 1], (100, 1))  # 外圈 + 滚动体

# 源域：用于大批量数据预训练 (单故障 + 正常)
X_source = np.concatenate((X_normal, X_inner, X_outer, X_rolling), axis=0)
y_source = np.concatenate((y_normal, y_inner, y_outer, y_rolling), axis=0)

# 目标域：用于极少量数据微调及后续测试 (复合故障)
X_target = np.concatenate((X_io, X_or), axis=0)
y_target = np.concatenate((y_io, y_or), axis=0)


print(f"源域数据形状 (X_source): {X_source.shape}, 标签形状: {y_source.shape}")
print(f"目标域数据形状 (X_target): {X_target.shape}, 标签形状: {y_target.shape}")

正在生成时频特征图...
源域数据形状 (X_source): (400, 3, 64, 48), 标签形状: (400, 3)
目标域数据形状 (X_target): (200, 3, 64, 48), 标签形状: (200, 3)


In [26]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# --- 源域划分 ---
# 论文中明确源域按照 8:2 划分为训练集和测试集 
X_src_train, X_src_test, y_src_train, y_src_test = train_test_split(
    X_source, y_source, test_size=0.2, random_state=42, stratify=y_source
)

# --- 目标域划分 ---
# 目标域是小样本迁移的核心。论文中提到随机抽取少量复合故障样本进行微调 [cite: 351, 583]。
# 我们假设总共抽取 20 个复合故障样本（内圈+外圈 10个，外圈+滚动体 10个）用于微调模型，剩余的用于最终测试。
X_tgt_tune, X_tgt_test, y_tgt_tune, y_tgt_test = train_test_split(
    X_target, y_target, train_size=20, random_state=42, stratify=y_target
)

# 将 NumPy 数组转换为 float32 类型的张量，这是 PyTorch 神经网络默认的数据类型
tensor_X_src_train = torch.tensor(X_src_train, dtype=torch.float32)
tensor_y_src_train = torch.tensor(y_src_train, dtype=torch.float32)

tensor_X_src_test = torch.tensor(X_src_test, dtype=torch.float32)
tensor_y_src_test = torch.tensor(y_src_test, dtype=torch.float32)

tensor_X_tgt_tune = torch.tensor(X_tgt_tune, dtype=torch.float32)
tensor_y_tgt_tune = torch.tensor(y_tgt_tune, dtype=torch.float32)

tensor_X_tgt_test = torch.tensor(X_tgt_test, dtype=torch.float32)
tensor_y_tgt_test = torch.tensor(y_tgt_test, dtype=torch.float32)

dataset_src_train = TensorDataset(tensor_X_src_train, tensor_y_src_train)
dataset_src_test = TensorDataset(tensor_X_src_test, tensor_y_src_test)
dataset_tgt_tune = TensorDataset(tensor_X_tgt_tune, tensor_y_tgt_tune)
dataset_tgt_test = TensorDataset(tensor_X_tgt_test, tensor_y_tgt_test)

# 论文中 Adam 优化器的 Batch Size 设置为 20 [cite: 622]
batch_size = 20

# 训练集和微调集需要打乱顺序 (shuffle=True)
loader_src_train = DataLoader(dataset_src_train, batch_size=batch_size, shuffle=True)
loader_tgt_tune = DataLoader(dataset_tgt_tune, batch_size=batch_size, shuffle=True)

# 测试集不需要打乱顺序 (shuffle=False)
loader_src_test = DataLoader(dataset_src_test, batch_size=batch_size, shuffle=False)
loader_tgt_test = DataLoader(dataset_tgt_test, batch_size=batch_size, shuffle=False)

print(f"源域训练集 Batch 数量: {len(loader_src_train)}")
print(f"目标域微调集 Batch 数量: {len(loader_tgt_tune)}")
print("DataLoader 封装完成！")

源域训练集 Batch 数量: 16
目标域微调集 Batch 数量: 1
DataLoader 封装完成！
